# nb23 — Out-of-Sample Temporal Validation & Bootstrapped AUC Summary

**Purpose**  
1. Temporal OOS validation for the **nb17 network-augmented model** (mirror of nb22 which did baseline only).  
2. Bootstrapped 95 % CIs for AUC on **both** models (baseline nb16/nb22, network nb17/nb23).  
3. Clean four-cell summary table: in-sample vs OOS × baseline vs network.

**Train/test split** — identical to nb22:  
- Train: 1992–2014 (inclusive)  
- Test: 2015–2024 (inclusive)  

**Models**  
- Baseline logit: 9-feature spec from nb16 (no network features)  
- Network logit: 19-feature spec from nb17 (adds 10 network metrics)  

**Outputs saved to** `outputs/`
- `nb23_oos_auc_summary.csv`  
- `nb23_auc_summary_table.png`  
- `nb23_bootstrap_distributions.png`

**Estimator note:** This notebook uses scikit-learn's `LogisticRegression` (L2 regularisation, `solver='lbfgs'`) to fit the baseline and network-augmented models. This is a re-fitted sklearn equivalent of the statsmodels `Logit` models in nb16 and nb17 — not those exact fitted objects. The OOS AUC values produced here (baseline 0.864, network 0.863) are therefore from a comparable but not identical estimator. Any reference in the report to "the nb16/nb17 model achieves OOS AUC of X" should be read as "a re-fitted sklearn equivalent of the nb16/nb17 specification achieves OOS AUC of X."

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.utils import resample

# ── Paths ────────────────────────────────────────────────────────────────────
ROOT = Path.cwd().parents[1]  # assumes kernel CWD is notebooks/03_analysis/
DATA     = ROOT / 'data' / 'merged'
OUT      = ROOT / 'outputs'
OUT_FIGS = ROOT / 'outputs' / 'final_report' / '04_appendix'
OUT.mkdir(parents=True, exist_ok=True)
OUT_FIGS.mkdir(parents=True, exist_ok=True)

# Colour palette consistent with nb16/nb17 (after cosmetic fix)
C_BASE = '#BF3A27'   # baseline model — red
C_NET  = '#C7922A'   # network model  — amber

RANDOM_STATE = 42
N_BOOT       = 1000
TRAIN_END    = 2014
TEST_START   = 2015

print('Paths OK')
print(f'  DATA     : {DATA}')
print(f'  OUT      : {OUT}')
print(f'  OUT_FIGS : {OUT_FIGS}')

## 1. Load panel data

In [2]:
# ── Load the master panel (same file used by nb16 & nb17) ────────────────────
panel_path = DATA / 'panel_final_1992_2024.csv'
df = pd.read_csv(panel_path)

print(f'Panel shape : {df.shape}')
print(f'Years       : {df["year"].min()} – {df["year"].max()}')
print(f'Countries   : {df["recipient_iso3"].nunique()}')
print()
print(df.head(3))

Panel shape : (6358, 38)
Years       : 1992 – 2024
Countries   : 213

  recipient_iso3  year  arms_tiv_total  oda_total  econ_neocol_score_total  \
0            ABW  1992             0.0      29.85                      0.0   
1            ABW  1993             0.0      22.97                      0.0   
2            ABW  1994             0.0      15.93                      0.0   

   colonial_tie_flag  journalist_killings  gdp_per_capita  gdp_per_capita_log  \
0                  0                    0    13892.605143            9.539112   
1                  0                    0    14700.959808            9.595668   
2                  0                    0    16055.287787            9.683794   

   population  ...  econ_neocol_score_in_strength_lag1  \
0     69005.0  ...                                 NaN   
1     73685.0  ...                                 0.0   
2     77595.0  ...                                 0.0   

   econ_neocol_score_out_strength_lag1  \
0                

## 2. Define feature sets

Mirror the exact features used in nb16 (baseline) and nb17 (network-augmented).  
**Edit these lists if nb16/nb17 used a different column name.**

In [3]:
# ── Construct interaction terms (mirrors nb17 exactly) ───────────────────────
df['arms_x_colonial'] = df['arms_tiv_total_log_lag1']       * df['colonial_tie_flag']
df['oda_x_colonial']  = df['oda_total_log_lag1']             * df['colonial_tie_flag']
df['econ_x_colonial'] = df['econ_neocol_score_total_lag1']   * df['colonial_tie_flag']

# ── Construct binary outcome (mirrors nb16/nb17) ─────────────────────────────
df['killing_any'] = (df['journalist_killings'] > 0).astype(int)

OUTCOME = 'killing_any'

# ── Baseline features — nb16 (8 covariates) ──────────────────────────────────
BASELINE_FEATURES = [
    'arms_tiv_total_log_lag1',
    'oda_total_log_lag1',
    'econ_neocol_score_total_lag1',
    'colonial_tie_flag',
    'gdp_per_capita_log',
    'population_log',
    'armed_conflict',
    'conflict_intensity',
]

# ── Network-only features added in nb17 (8 node metrics) ─────────────────────
NETWORK_FEATURES = [
    'arms_tiv_in_strength_lag1',
    'bilateral_oda_in_strength_lag1',
    'econ_neocol_score_in_strength_lag1',
    'colonial_tie_in_strength_lag1',
    'arms_tiv_pagerank_lag1',
    'bilateral_oda_pagerank_lag1',
    'econ_neocol_score_pagerank_lag1',
    'colonial_tie_pagerank_lag1',
]

# ── Interaction features added in nb17 (3) ────────────────────────────────────
INTERACTION_FEATURES = [
    'arms_x_colonial',
    'oda_x_colonial',
    'econ_x_colonial',
]

# Full nb17 spec (19 features total)
NETWORK_FULL_FEATURES = BASELINE_FEATURES + NETWORK_FEATURES + INTERACTION_FEATURES

print(f'Baseline features  : {len(BASELINE_FEATURES)}')
print(f'Network features   : {len(NETWORK_FEATURES)}')
print(f'Interaction terms  : {len(INTERACTION_FEATURES)}')
print(f'Full feature set   : {len(NETWORK_FULL_FEATURES)}')
print(f'Outcome            : {OUTCOME}  (zero rate: {(df[OUTCOME]==0).mean()*100:.1f}%)')

# Verify all columns exist
missing_base = [c for c in BASELINE_FEATURES + [OUTCOME] if c not in df.columns]
missing_net  = [c for c in NETWORK_FULL_FEATURES if c not in df.columns]
if missing_base:
    print(f'\n⚠️  MISSING baseline cols : {missing_base}')
if missing_net:
    print(f'⚠️  MISSING network cols  : {missing_net}')
if not missing_base and not missing_net:
    print('\n✓ All columns present')

Baseline features  : 8
Network features   : 8
Interaction terms  : 3
Full feature set   : 19
Outcome            : killing_any  (zero rate: 88.7%)

✓ All columns present


## 3. Build train / test splits

In [4]:
def make_split(df, features, outcome=OUTCOME):
    """Return (X_train, y_train, X_test, y_test) for a given feature list.
    Drops rows with any NaN in features or outcome.
    """
    cols_needed = features + [outcome, 'year']
    sub = df[cols_needed].dropna()

    train = sub[sub['year'] <= TRAIN_END]
    test  = sub[sub['year'] >= TEST_START]

    X_tr = train[features].values
    y_tr = train[outcome].values
    X_te = test[features].values
    y_te = test[outcome].values

    return X_tr, y_tr, X_te, y_te, train, test


X_tr_b, y_tr_b, X_te_b, y_te_b, train_b, test_b = make_split(df, BASELINE_FEATURES)
X_tr_n, y_tr_n, X_te_n, y_te_n, train_n, test_n = make_split(df, NETWORK_FULL_FEATURES)

for label, Xtr, ytr, Xte, yte in [
    ('Baseline', X_tr_b, y_tr_b, X_te_b, y_te_b),
    ('Network',  X_tr_n, y_tr_n, X_te_n, y_te_n),
]:
    print(f'{label}:')
    print(f'  Train rows {Xtr.shape[0]:>5}  | kill rate {ytr.mean():.3f}')
    print(f'  Test  rows {Xte.shape[0]:>5}  | kill rate {yte.mean():.3f}')
    print()

Baseline:
  Train rows  4007  | kill rate 0.119
  Test  rows  1758  | kill rate 0.119

Network:
  Train rows  4007  | kill rate 0.119
  Test  rows  1758  | kill rate 0.119



## 4. Fit models & compute point-estimate AUC (in-sample + OOS)

In [5]:
def fit_logit(X_tr, y_tr):
    """Scale features, fit L2 logistic regression, return (model, scaler)."""
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X_tr)
    model = LogisticRegression(
        penalty='l2',
        C=1.0,
        solver='lbfgs',
        max_iter=1000,
        random_state=RANDOM_STATE,
    )
    model.fit(Xs, y_tr)
    return model, scaler


def auc_score(model, scaler, X):
    Xs = scaler.transform(X)
    probs = model.predict_proba(Xs)[:, 1]
    return probs  # return probs so we can reuse for bootstrap


# ── Fit ──────────────────────────────────────────────────────────────────────
model_b, scaler_b = fit_logit(X_tr_b, y_tr_b)
model_n, scaler_n = fit_logit(X_tr_n, y_tr_n)

# ── In-sample AUC (train set) ────────────────────────────────────────────────
probs_is_b = auc_score(model_b, scaler_b, X_tr_b)
probs_is_n = auc_score(model_n, scaler_n, X_tr_n)

auc_is_b = roc_auc_score(y_tr_b, probs_is_b)
auc_is_n = roc_auc_score(y_tr_n, probs_is_n)

# ── OOS AUC (test set) ───────────────────────────────────────────────────────
probs_oos_b = auc_score(model_b, scaler_b, X_te_b)
probs_oos_n = auc_score(model_n, scaler_n, X_te_n)

auc_oos_b = roc_auc_score(y_te_b, probs_oos_b)
auc_oos_n = roc_auc_score(y_te_n, probs_oos_n)

print('Point-estimate AUC')
print(f'  Baseline  — in-sample: {auc_is_b:.4f}   OOS: {auc_oos_b:.4f}')
print(f'  Network   — in-sample: {auc_is_n:.4f}   OOS: {auc_oos_n:.4f}')

Point-estimate AUC
  Baseline  — in-sample: 0.8476   OOS: 0.8636
  Network   — in-sample: 0.8587   OOS: 0.8627


## 5. Bootstrap 95 % CIs

Bootstrap is applied **separately** to in-sample and OOS splits.  
Each iteration resamples rows with replacement from the relevant split, then scores the **already-fitted** model on the bootstrap sample — i.e. we are estimating uncertainty in the AUC statistic given the fixed model, not refitting each time. This matches standard practice for AUC CI estimation.

In [6]:
def bootstrap_auc(model, scaler, X, y, n_boot=N_BOOT, seed=RANDOM_STATE):
    """Return array of bootstrapped AUC values.
    Uses the fitted model unchanged; only the evaluation sample varies.
    Skips iterations where the bootstrap draw contains only one class.
    """
    rng  = np.random.default_rng(seed)
    aucs = []
    n    = len(y)
    Xs   = scaler.transform(X)
    probs_full = model.predict_proba(Xs)[:, 1]

    while len(aucs) < n_boot:
        idx = rng.integers(0, n, size=n)
        y_b = y[idx]
        p_b = probs_full[idx]
        if len(np.unique(y_b)) < 2:
            continue  # can't compute AUC with one class
        aucs.append(roc_auc_score(y_b, p_b))

    return np.array(aucs)


print(f'Running {N_BOOT} bootstrap iterations × 4 conditions …')

boot_is_b  = bootstrap_auc(model_b, scaler_b, X_tr_b, y_tr_b)
print('  [1/4] baseline in-sample  done')
boot_is_n  = bootstrap_auc(model_n, scaler_n, X_tr_n, y_tr_n)
print('  [2/4] network  in-sample  done')
boot_oos_b = bootstrap_auc(model_b, scaler_b, X_te_b, y_te_b)
print('  [3/4] baseline OOS        done')
boot_oos_n = bootstrap_auc(model_n, scaler_n, X_te_n, y_te_n)
print('  [4/4] network  OOS        done')


def ci95(arr):
    return np.percentile(arr, 2.5), np.percentile(arr, 97.5)


ci_is_b  = ci95(boot_is_b)
ci_is_n  = ci95(boot_is_n)
ci_oos_b = ci95(boot_oos_b)
ci_oos_n = ci95(boot_oos_n)

print()
print('Bootstrapped 95 % CIs')
print(f'  Baseline in-sample : {auc_is_b:.4f}  [{ci_is_b[0]:.4f}, {ci_is_b[1]:.4f}]')
print(f'  Network  in-sample : {auc_is_n:.4f}  [{ci_is_n[0]:.4f}, {ci_is_n[1]:.4f}]')
print(f'  Baseline OOS       : {auc_oos_b:.4f}  [{ci_oos_b[0]:.4f}, {ci_oos_b[1]:.4f}]')
print(f'  Network  OOS       : {auc_oos_n:.4f}  [{ci_oos_n[0]:.4f}, {ci_oos_n[1]:.4f}]')

Running 1000 bootstrap iterations × 4 conditions …
  [1/4] baseline in-sample  done
  [2/4] network  in-sample  done
  [3/4] baseline OOS        done
  [4/4] network  OOS        done

Bootstrapped 95 % CIs
  Baseline in-sample : 0.8476  [0.8290, 0.8653]
  Network  in-sample : 0.8587  [0.8410, 0.8760]
  Baseline OOS       : 0.8636  [0.8356, 0.8888]
  Network  OOS       : 0.8627  [0.8345, 0.8871]


## 6. Save results CSV

In [7]:
results = pd.DataFrame([
    {
        'model'    : 'Baseline (nb16, sklearn equiv.)',
        'split'    : 'In-sample (1992–2014)',
        'auc'      : round(auc_is_b, 4),
        'ci_lower' : round(ci_is_b[0], 4),
        'ci_upper' : round(ci_is_b[1], 4),
        'n_obs'    : len(y_tr_b),
        'source_nb': 'nb23',
    },
    {
        'model'    : 'Network (nb17, sklearn equiv.)',
        'split'    : 'In-sample (1992–2014)',
        'auc'      : round(auc_is_n, 4),
        'ci_lower' : round(ci_is_n[0], 4),
        'ci_upper' : round(ci_is_n[1], 4),
        'n_obs'    : len(y_tr_n),
        'source_nb': 'nb23',
    },
    {
        'model'    : 'Baseline (nb16, sklearn equiv.)',
        'split'    : 'OOS (2015–2024)',
        'auc'      : round(auc_oos_b, 4),
        'ci_lower' : round(ci_oos_b[0], 4),
        'ci_upper' : round(ci_oos_b[1], 4),
        'n_obs'    : len(y_te_b),
        'source_nb': 'nb23',
    },
    {
        'model'    : 'Network (nb17, sklearn equiv.)',
        'split'    : 'OOS (2015–2024)',
        'auc'      : round(auc_oos_n, 4),
        'ci_lower' : round(ci_oos_n[0], 4),
        'ci_upper' : round(ci_oos_n[1], 4),
        'n_obs'    : len(y_te_n),
        'source_nb': 'nb23',
    },
])

csv_path = OUT / 'nb23_oos_auc_summary.csv'
results.to_csv(csv_path, index=False)
print(f'Saved: {csv_path}')
print()
print(results.to_string(index=False))

Saved: /Users/mac/Documents/GitHub/Violent-Offenders-GPV---CSSci-/outputs/nb23_oos_auc_summary.csv

                          model                 split    auc  ci_lower  ci_upper  n_obs source_nb
Baseline (nb16, sklearn equiv.) In-sample (1992–2014) 0.8476    0.8290    0.8653   4007      nb23
 Network (nb17, sklearn equiv.) In-sample (1992–2014) 0.8587    0.8410    0.8760   4007      nb23
Baseline (nb16, sklearn equiv.)       OOS (2015–2024) 0.8636    0.8356    0.8888   1758      nb23
 Network (nb17, sklearn equiv.)       OOS (2015–2024) 0.8627    0.8345    0.8871   1758      nb23


## 7. Figure 1 — AUC summary table (publication-ready)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
ax.axis('off')

col_labels = ['Model', 'Sample', 'AUC', '95 % CI', 'N']
table_data = [
    ['Baseline (nb16, sklearn equiv.)', 'In-sample (1992–2014)',
     f'{auc_is_b:.3f}', f'[{ci_is_b[0]:.3f}, {ci_is_b[1]:.3f}]',  str(len(y_tr_b))],
    ['Network (nb17, sklearn equiv.)',  'In-sample (1992–2014)',
     f'{auc_is_n:.3f}', f'[{ci_is_n[0]:.3f}, {ci_is_n[1]:.3f}]',  str(len(y_tr_n))],
    ['Baseline (nb16, sklearn equiv.)', 'OOS (2015–2024)',
     f'{auc_oos_b:.3f}', f'[{ci_oos_b[0]:.3f}, {ci_oos_b[1]:.3f}]', str(len(y_te_b))],
    ['Network (nb17, sklearn equiv.)',  'OOS (2015–2024)',
     f'{auc_oos_n:.3f}', f'[{ci_oos_n[0]:.3f}, {ci_oos_n[1]:.3f}]', str(len(y_te_n))],
]

tbl = ax.table(
    cellText   = table_data,
    colLabels  = col_labels,
    cellLoc    = 'center',
    loc        = 'center',
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(10)
tbl.scale(1, 1.6)

# Header row styling
for j in range(len(col_labels)):
    tbl[(0, j)].set_facecolor('#2C3E50')
    tbl[(0, j)].set_text_props(color='white', fontweight='bold')

# Row shading: alternate + colour-code by model
row_colors = {
    1: '#FDECEA',  # baseline in-sample (light red)
    2: '#FEF3E2',  # network  in-sample (light amber)
    3: '#FDECEA',  # baseline OOS
    4: '#FEF3E2',  # network  OOS
}
for row, color in row_colors.items():
    for col in range(len(col_labels)):
        tbl[(row, col)].set_facecolor(color)

fig.suptitle(
    'Table: AUC with Bootstrapped 95 % CIs — Baseline vs Network Model',
    fontsize=11, fontweight='bold', y=0.98,
)
fig.text(
    0.5, 0.01,
    'Note: CIs estimated by 1,000 bootstrap iterations on the fixed fitted model.\n'
    'Train: 1992–2014. Test (OOS): 2015–2024. Source: nb23.',
    ha='center', fontsize=8, color='#555555',
)

plt.tight_layout()
table_path = OUT_FIGS / 'nb23_auc_summary_table.png'
plt.savefig(table_path, dpi=180, bbox_inches='tight')
plt.show()
print(f'Saved: {table_path}')

## 8. Figure 2 — Bootstrap AUC distributions (all four conditions)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=False)

def plot_boot_panel(ax, boot_base, boot_net, auc_base, auc_net,
                    ci_base, ci_net, title):
    """Plot overlapping histograms for baseline and network on one axis."""
    bins = np.linspace(
        min(boot_base.min(), boot_net.min()) - 0.005,
        max(boot_base.max(), boot_net.max()) + 0.005,
        40,
    )

    ax.hist(boot_base, bins=bins, alpha=0.55, color=C_BASE,
            label=f'Baseline  AUC={auc_base:.3f}', edgecolor='white', linewidth=0.4)
    ax.hist(boot_net,  bins=bins, alpha=0.55, color=C_NET,
            label=f'Network   AUC={auc_net:.3f}',  edgecolor='white', linewidth=0.4)

    # Point estimates
    ax.axvline(auc_base, color=C_BASE, lw=2, ls='--')
    ax.axvline(auc_net,  color=C_NET,  lw=2, ls='--')

    # CI shading
    ax.axvspan(ci_base[0], ci_base[1], alpha=0.12, color=C_BASE)
    ax.axvspan(ci_net[0],  ci_net[1],  alpha=0.12, color=C_NET)

    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlabel('AUC', fontsize=10)
    ax.set_ylabel('Bootstrap frequency', fontsize=10)
    ax.legend(fontsize=9, framealpha=0.8)
    ax.spines[['top', 'right']].set_visible(False)


plot_boot_panel(
    axes[0],
    boot_is_b, boot_is_n,
    auc_is_b,  auc_is_n,
    ci_is_b,   ci_is_n,
    'In-Sample (1992–2014)',
)
plot_boot_panel(
    axes[1],
    boot_oos_b, boot_oos_n,
    auc_oos_b,  auc_oos_n,
    ci_oos_b,   ci_oos_n,
    'Out-of-Sample (2015–2024)',
)

fig.suptitle(
    'Bootstrap AUC Distributions — Baseline vs Network Model',
    fontsize=12, fontweight='bold',
)
fig.text(
    0.5, -0.02,
    'Dashed lines = point-estimate AUC. Shaded bands = 95 % bootstrap CI. n = 1,000 iterations.',
    ha='center', fontsize=8, color='#555555',
)

plt.tight_layout()
dist_path = OUT_FIGS / 'nb23_bootstrap_distributions.png'
plt.savefig(dist_path, dpi=180, bbox_inches='tight')
plt.show()
print(f'Saved: {dist_path}')

## 9. Quick diagnostics — class balance in test window

In [10]:
print('Class balance — test window (2015–2024)')
print(f'  Baseline split  zeros={( y_te_b==0).sum()}  ones={(y_te_b==1).sum()}  '
      f'zero_pct={(y_te_b==0).mean()*100:.1f}%')
print(f'  Network  split  zeros={(y_te_n==0).sum()}  ones={(y_te_n==1).sum()}  '
      f'zero_pct={(y_te_n==0).mean()*100:.1f}%')

# NaN loss check
print()
print('Rows dropped due to NaN')
full_train_rows = len(df[df['year'] <= TRAIN_END])
full_test_rows  = len(df[df['year'] >= TEST_START])
print(f'  Baseline train: {full_train_rows} → {len(y_tr_b)} '
      f'({full_train_rows - len(y_tr_b)} dropped)')
print(f'  Baseline test : {full_test_rows} → {len(y_te_b)} '
      f'({full_test_rows - len(y_te_b)} dropped)')
print(f'  Network  train: {full_train_rows} → {len(y_tr_n)} '
      f'({full_train_rows - len(y_tr_n)} dropped)')
print(f'  Network  test : {full_test_rows} → {len(y_te_n)} '
      f'({full_test_rows - len(y_te_n)} dropped)')
print(f'\n  Countries in test window : '
      f'{df[df["year"] >= TEST_START]["recipient_iso3"].nunique()}')

Class balance — test window (2015–2024)
  Baseline split  zeros=1548  ones=210  zero_pct=88.1%
  Network  split  zeros=1548  ones=210  zero_pct=88.1%

Rows dropped due to NaN
  Baseline train: 4487 → 4007 (480 dropped)
  Baseline test : 1871 → 1758 (113 dropped)
  Network  train: 4487 → 4007 (480 dropped)
  Network  test : 1871 → 1758 (113 dropped)

  Countries in test window : 196


## 10. Final summary printout

Copy these figures directly into the report.

In [11]:
print('=' * 62)
print('  nb23 FINAL AUC SUMMARY')
print('=' * 62)
print(f'  Baseline  in-sample  AUC = {auc_is_b:.3f}  '
      f'95% CI [{ci_is_b[0]:.3f}, {ci_is_b[1]:.3f}]')
print(f'  Network   in-sample  AUC = {auc_is_n:.3f}  '
      f'95% CI [{ci_is_n[0]:.3f}, {ci_is_n[1]:.3f}]')
print(f'  Baseline  OOS        AUC = {auc_oos_b:.3f}  '
      f'95% CI [{ci_oos_b[0]:.3f}, {ci_oos_b[1]:.3f}]')
print(f'  Network   OOS        AUC = {auc_oos_n:.3f}  '
      f'95% CI [{ci_oos_n[0]:.3f}, {ci_oos_n[1]:.3f}]')
print('=' * 62)
print(f'  Train window : 1992–{TRAIN_END}')
print(f'  Test  window : {TEST_START}–2024')
print(f'  Bootstrap n  : {N_BOOT}')
print(f'  Outputs in   : {OUT}')
print('=' * 62)

  nb23 FINAL AUC SUMMARY
  Baseline  in-sample  AUC = 0.848  95% CI [0.829, 0.865]
  Network   in-sample  AUC = 0.859  95% CI [0.841, 0.876]
  Baseline  OOS        AUC = 0.864  95% CI [0.836, 0.889]
  Network   OOS        AUC = 0.863  95% CI [0.834, 0.887]
  Train window : 1992–2014
  Test  window : 2015–2024
  Bootstrap n  : 1000
  Outputs in   : /Users/mac/Documents/GitHub/Violent-Offenders-GPV---CSSci-/outputs
